Now that we have shown that the k-fold models generalize better to the test data than a random baseline, we will generate a new model with the best hyperparameters identified, but trained on the training + validation data. This is simply to have one coherent model for additional downstream assessments. It also should give the model better power.

In [24]:
import os

import pandas as pd
import scanpy as sc

import torch
import torch.nn as nn
import numpy as np

In [12]:
import sys
sclembas = '/Users/hratchbaghdassarian/Downloads/scLEMBAS'#'/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.train import TrainSC


In [13]:
seed = 888
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/Users/hratchbaghdassarian/Downloads/sclembas_data'#'/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')

In [14]:
adata = sc.read_h5ad(os.path.join(data_path, 'processed', 'kang_expr_scored.h5ad'))
sn_ppis = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_sn_ppis.csv'), index_col = 0)

tf_adata = io.read_tfad(os.path.join(data_path, 'processed', 'Kang_tf_activity.h5ad'))
tf_adata.obs['condition'] = tf_adata.obs['stim'].astype(str) + '^' + tf_adata.obs['seurat_annotations'].astype(str)

In [15]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [17]:
val_res = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'), index_col = 0)

avg_loss = val_res.groupby(['max_epochs', 'max_lr', 'train_batch_size'])['emd_loss_total'].mean()
min_loss = avg_loss.min()
best_params = avg_loss[avg_loss == min_loss]


best_params = val_res[(val_res.max_epochs == best_params.index[0][0]) &
                      (val_res.max_lr == best_params.index[0][1]) &
                      (val_res.train_batch_size == best_params.index[0][2])]

untrained = best_params[best_params.emd_loss_total.isna()]
if untrained.shape[0] > 0:
    print('The following models did not converge: k = ' + ', '.join([str(k) for k in untrained.k.tolist()]))

best_params = best_params[best_params.emd_loss_total.notna()].reset_index(drop = True)
if best_params.KL_regularization.nunique() != 1:
    raise ValueError("Haven't prepared for the scenario where multiple KL regularization values were used")

The following models did not converge: k = 2.0


<span style="color:red">TODO: double check that the trainer_k hyperparameters are the same across the folds</span>


In [41]:
# these should have the same hyper parameters
trainer_og = trainers_k[best_params.k.tolist()[0]]
mod_og = trainer_og.mod

<span style="color:red">TODO: check that these train cells are the union of the training and validation across the 5-folds</span>


In [34]:
test_cells = os.path.join(data_path, 'processed', 'data_split_barcodes', 'kang_test.txt').read().splitlines()
train_cells = set(tf_adata.obs_names).difference(test_cells)

Build and train the full model:

In [ ]:
mod_full = SignalingModel(net = sn_ppis,
                     X_in = mod_og.X_in.copy(),
                     y_out = mod_og.y_out.copy(), 
                     expr = mod_og.expr.copy(), 
                     covariates = mod_og.signaling_network.covariates.copy(),
                     categorical_covariate_keys = mod_og.signaling_network.covariates.columns.tolist(),
                     projection_amplitude_in = mod_og.input_layer.projection_amplitude, 
                     projection_amplitude_out = mod_og.projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = mod_og.signaling_network.bionet_params.copy(), 
                     dtype = torch.float32, device = device, seed = seed)
mod_full.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
mod_full.signaling_network.prescale_weights(target_radius = mod_og.signaling_network.bionet_params['spectral_target']) # spectral radius

trainer = TrainSC(mod = mod_full,
                  prediction_optimizer = torch.optim.Adam,
                  prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device),
                  discriminator_params = trainer_og.discriminator['params'].copy(),
                  hyper_params = trainer_og.hyper_params.copy(),
              train_split = {'train': train_cells, 'test': test_cells, 'validation': None},
              train_seed = seed,
              track_test = True,
              track_validation = False)

mod_full = trainer.train_model(verbose = False)